# 04. CLIP Baseline (Multimodal Image-Text Search)
**Seksi 8 & Seksi 11 (Eksperimen E4)** pada `instruction.md`

### Tujuan:
1. Membangun pencarian silang modalitas (Text-to-Image & Image-to-Image) menggunakan model pretrained CLIP (`clip-ViT-B-32`).
2. Memvalidasi kemampuan pengguna mencari barang dengan foto referensi atau teks kueri yang langsung mencocokkan foto fisik di gudang.

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/processed/inventory_catalog.csv')
img_base = '../../project/dataset/barudownload/retail-products-classification/'
df['full_img_path'] = df['image_path'].apply(lambda x: os.path.join(img_base, x))

print(f'Total Item dengan Foto Tersedia: {len(df)}')
df[['item_id', 'item_name', 'category', 'image_path']].head(3)

C:\Users\farha\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total Item dengan Foto Tersedia: 2100


,item_id,item_name,category,image_path
0,B005NIX12S,Bootstraps Tourquoise Polka (Pair),All Beauty,train/train/B005NIX12S.jpg
1,B001LBXBJY,Laura Mercier Lip Colour Hydrangea ( Creme ) 3...,All Beauty,train/train/B001LBXBJY.jpg
2,B005WKXAFA,NYC New York Color Smooch Proof Long Wearing L...,All Beauty,train/train/B005WKXAFA.jpg


In [2]:
# Memuat model CLIP multimodal
clip_model = SentenceTransformer('clip-ViT-B-32')
print('Model CLIP-ViT-B-32 berhasil dimuat!')

# Ekstraksi Text Embedding
text_embeddings = clip_model.encode(df['item_name'].tolist(), batch_size=32, normalize_embeddings=True, show_progress_bar=True)
print(f'Dimensi Text Embedding CLIP: {text_embeddings.shape}')

C:\Users\farha\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\farha\.cache\huggingface\hub\models--sentence-transformers--clip-ViT-B-32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\farha\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_downlo

Model CLIP-ViT-B-32 berhasil dimuat!


Batches: 100%|██████████| 66/66 [01:24<00:00,  1.27s/it]

Dimensi Text Embedding CLIP: (2100, 512)


In [3]:
def search_clip_text(query, top_k=3):
    q_emb = clip_model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, text_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    
    res = []
    for rank, idx in enumerate(top_idx, 1):
        r = df.iloc[idx]
        res.append({
            'Rank': rank,
            'Item': r['item_name'],
            'Category': r['category'],
            'Similarity': round(float(sims[idx]), 4),
            'Photo': r['image_path']
        })
    return pd.DataFrame(res)

search_clip_text('stainless steel screwdriver for electronics', top_k=3)

,Rank,Item,Category,Similarity,Photo
0,1,Titanium Solder Pick,"Arts, Crafts & Sewing",0.7937,train/train/B000HYQI3W.jpg
1,2,Cigarette Lighter/Jack Assembly,Automotive,0.7819,train/train/B0002KR1Z0.jpg
2,3,Allparts Nickel String Guides,Musical Instruments,0.7729,train/train/B0002D0H4Y.jpg
